# Form checker (check_form)

Reads a loan application form image with DeepSeek-OCR when it is available, otherwise Tesseract.
Extracts the fields and compares them with `payroll.csv`. Each run is appended to `decisions.jsonl`,
including which OCR engine was used.

Decisions:
- missing required fields, a stale date (over 183 days), or an unknown ID: **uncertain**
- name or income differs from payroll: **reject**
- everything matches: **clear**

To test another form, change `FORM_PATH` in the run cell.

In [ ]:
import csv
import json
import os
import re
import tempfile
from datetime import date, datetime, timedelta
from pathlib import Path

import cv2
import pytesseract


class OCREngine:
    TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    DEFAULT_MODEL_DIR = r"C:\Users\PCD\OneDrive\Apps\loan-agent"
    DEEPSEEK_PROMPT = "<image>\n<|grounding|>Convert the document to markdown."

    def __init__(self, backend=None, model_dir=None):
        self.requested = backend or os.environ.get("LOAN_OCR_BACKEND", "auto")
        self.model_dir = model_dir or os.environ.get("LOAN_DEEPSEEK_DIR", self.DEFAULT_MODEL_DIR)
        pytesseract.pytesseract.tesseract_cmd = self.TESSERACT_CMD
        self._tok = None
        self._model = None
        self.backend, self.reason = self._choose()
        print(f"OCR engine: {self.backend} ({self.reason})")

    def _unavailable_reason(self):
        if not os.path.isdir(self.model_dir) and "/" not in self.model_dir:
            return f"model folder not found: {self.model_dir}"
        try:
            import torch
        except ImportError:
            return "torch is not installed"
        if not torch.cuda.is_available():
            return "no CUDA GPU detected"
        try:
            import transformers  # noqa: F401
        except ImportError:
            return "transformers is not installed"
        return None

    def _choose(self):
        if self.requested == "tesseract":
            return "tesseract", "set to tesseract"
        if self.requested == "deepseek":
            reason = self._unavailable_reason()
            if reason:
                raise RuntimeError(f"DeepSeek requested but unavailable: {reason}")
            return "deepseek", "set to deepseek"
        if self.requested == "auto":
            reason = self._unavailable_reason()
            if reason is None:
                return "deepseek", "GPU and model available"
            return "tesseract", f"fallback: {reason}"
        raise ValueError(f"Unknown backend: {self.requested!r}")

    def _load_deepseek(self):
        if self._model is None:
            import torch
            from transformers import AutoModel, AutoTokenizer

            self._tok = AutoTokenizer.from_pretrained(self.model_dir, trust_remote_code=True)
            self._model = AutoModel.from_pretrained(
                self.model_dir,
                trust_remote_code=True,
                use_safetensors=True,
                attn_implementation="eager",
            ).to(dtype=torch.bfloat16, device="cuda").eval()
        return self._tok, self._model

    def _read_tesseract(self, image_path):
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return pytesseract.image_to_string(gray)

    def _read_deepseek(self, image_path):
        import torch

        tok, model = self._load_deepseek()
        out_dir = tempfile.mkdtemp(prefix="deepseek_ocr_")
        with torch.inference_mode():
            model.infer(
                tok,
                prompt=self.DEEPSEEK_PROMPT,
                image_file=image_path,
                output_path=out_dir,
                base_size=1024,
                image_size=640,
                crop_mode=False,
                save_results=True,
                test_compress=True,
            )
        # The exact output filename is not confirmed yet, so collect any text file it wrote.
        texts = [
            p.read_text(encoding="utf-8", errors="ignore")
            for p in sorted(Path(out_dir).rglob("*"))
            if p.suffix in {".md", ".mmd", ".txt"}
        ]
        if not texts:
            raise RuntimeError(f"DeepSeek-OCR produced no text output in {out_dir}")
        return "\n".join(texts)

    def read_text(self, image_path):
        """Return the raw OCR text for one form image."""
        if self.backend == "deepseek":
            try:
                return self._read_deepseek(image_path)
            except Exception as exc:
                if self.requested == "deepseek":
                    raise
                print(f"DeepSeek failed on {image_path} ({exc}); using Tesseract for this form.")
        return self._read_tesseract(image_path)


ocr = OCREngine()


In [ ]:
PAYROLL_FILE = "payroll.csv"
LOG_FILE = "decisions.jsonl"
MAX_AGE = timedelta(days=183)  # about 6 months

ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

REQUIRED_FIELDS = ["national_id", "full_name", "declared_income", "form_date"]

FIELD_RULES = {
    "national_id": {
        "labels": [
            "ID / Iqama number", "National ID", "Iqama number", "ID number",
            "Iqama", "ID",
            "رقم الهوية / الإقامة", "رقم الهوية", "رقم الإقامة",
        ],
        "value": r"(\d{10})",
    },
    "full_name": {
        "labels": ["Full Name", "Applicant Name", "Customer Name", "Name"],
        "value": r"([^\n]+)",
    },
    "declared_income": {
        "labels": [
            "Total Monthly Income", "Monthly Income", "Total Income", "Income",
            "Monthly Salary", "Total Salary", "Net Salary", "Salary",
            "الدخل الشهري", "إجمالي الراتب", "الراتب",
        ],
        "value": r"([\d,]+(?:\.\d+)?)",
    },
    "form_date": {
        "labels": [
            "Form Date", "Issue Date", "Date of Issue", "Issued On",
            "تاريخ الإصدار", "تاريخ الاصدار",
        ],
        "value": r"(\d{4}[-/]\d{2}[-/]\d{2}|\d{1,2}[-/]\d{1,2}[-/]\d{4})",
    },
    "loan_type": {
        "labels": ["Loan Type", "Product", "Product Type", "نوع القرض"],
        "value": r"(Personal|Auto|Home|Business|شخصي|سيارات|عقاري|تجاري)",
    },
}


In [ ]:
def load_payroll(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return {row["national_id"]: row for row in csv.DictReader(f)}


def normalize(text):
    text = text.translate(ARABIC_DIGITS)
    return re.sub(r"[ \t]+", " ", text)


def find_field(text, rule):
    for label in rule["labels"]:
        pattern = (
            r"(?<![A-Za-z])" + re.escape(label)
            + r"\s*[:：\-]\s*" + rule["value"]
        )
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return None


def clean_name(name):
    if not name:
        return None
    cleaned = re.sub(r"\s+(Monthly|Income|Total|Form|Loan)\b.*$", "", name).strip()
    return cleaned or None


def parse_date(value):
    if not value:
        return None
    value = value.replace("/", "-")
    for fmt in ("%Y-%m-%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    return None


def is_stale(value):
    issued = parse_date(value)
    if issued is None:
        return None
    return (date.today() - issued) > MAX_AGE


def extract_fields(image_path):
    text = normalize(ocr.read_text(image_path))
    fields = {name: find_field(text, rule) for name, rule in FIELD_RULES.items()}
    fields["full_name"] = clean_name(fields["full_name"])
    return fields, text


def add_shadow_dbr(fields, payroll):
    """Shadow mode: record the system's obligations and the DBR. Never changes the decision."""
    nid = fields.get("national_id")
    record = payroll.get(nid)
    if record is None or not record.get("monthly_obligations"):
        fields["system_obligations"] = None
        fields["shadow_dbr"] = None
        return
    obligations = float(record["monthly_obligations"])
    declared = fields.get("declared_income")
    declared = float(declared.replace(",", "")) if declared else 0
    fields["system_obligations"] = obligations
    fields["shadow_dbr"] = round(obligations / declared, 4) if declared else None


def decide(fields, payroll):
    missing = [k for k in REQUIRED_FIELDS if not fields.get(k)]
    if missing:
        return "uncertain", f"Could not extract fields: {', '.join(missing)}"

    stale = is_stale(fields["form_date"])
    if stale is None:
        return "uncertain", f"Invalid form date: {fields['form_date']}"
    if stale:
        return "uncertain", f"Form dated {fields['form_date']} is older than 6 months"

    nid = fields["national_id"]
    if nid not in payroll:
        return "uncertain", f"National ID {nid} not found in payroll records"

    record = payroll[nid]

    if record["full_name"].lower() != fields["full_name"].lower():
        return "reject", (
            f"Name mismatch: form says '{fields['full_name']}', "
            f"payroll says '{record['full_name']}'"
        )

    declared = float(fields["declared_income"].replace(",", ""))
    actual = float(record["monthly_salary"])
    if abs(declared - actual) > 0.01:
        return "reject", (
            f"Income mismatch: form declares {declared:.0f}, "
            f"payroll shows {actual:.0f}"
        )

    return "clear", "All fields match payroll records"


def log_decision(form_path, decision, reason, fields):
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "form": form_path,
        "decision": decision,
        "reason": reason,
        "ocr_engine": ocr.last_engine or ocr.backend,
        "extracted": fields,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


## Run on one form

In [ ]:
FORM_PATH = "samples/form1.png"   # change this to test another form

payroll = load_payroll(PAYROLL_FILE)
fields, raw_text = extract_fields(FORM_PATH)
decision, reason = decide(fields, payroll)
add_shadow_dbr(fields, payroll)
log_decision(FORM_PATH, decision, reason, fields)

print(f"OCR engine: {ocr.last_engine or ocr.backend}")
print(f"Decision: {decision.upper()}")
print(f"Reason:   {reason}")
print(f"Fields:   {fields}")
if decision == "uncertain" and any(not fields.get(k) for k in REQUIRED_FIELDS):
    print("\nRaw OCR text for debugging:\n" + raw_text)
